In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import ( accuracy_score,precision_score,recall_score,f1_score,confusion_matrix,roc_auc_score,roc_curve)
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras import regularizers
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping
from tqdm.keras import TqdmCallback
import matplotlib.pyplot as plt

In [2]:
metrics = []
confusion_matrixs = []

In [3]:
def predict(model, data, threshold):
    reconstructions = model.predict(data)
    loss = tf.keras.losses.mae(reconstructions, data)
    return (loss.numpy() < threshold).astype(int)


def print_stats(predictions, labels):
    accuracy = accuracy_score(labels, predictions)
    precision = precision_score(labels, predictions)
    recall = recall_score(labels, predictions)
    f1 = f1_score(labels, predictions)
    print(f"Accuracy = {accuracy}")
    print(f"Precision = {precision}")
    print(f"Recall = {recall}")
    print(f"F1 score = {f1}")
    return accuracy, precision, recall, f1

In [8]:
data = pd.read_csv("Data/data 1.csv")

X = data.drop(columns=["Label"])
y = data["Label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
n_features = X.shape[1]

In [9]:
data.columns

Index(['0', '1', '2', '3', '4', '5', '6', '7', '8', '9',
       ...
       '386', '387', '388', '389', '390', '391', '392', '393', '394', 'Label'],
      dtype='object', length=360)

In [10]:
def create_autoencoder(input_dim, latent_dim=16):
    input_layer = Input(shape=(input_dim,), name="Input")
    encoded = Dense(
        128,
        activation="relu",
        activity_regularizer=regularizers.l1(10e-5),
        name="Encoding",
    )(input_layer)
    latent = Dense(16, activation="relu", name="Latent")(encoded)
    decoded = Dense(128, activation="relu", name="Decoding")(latent)
    output_layer = Dense(input_dim, activation="linear", name="Output")(decoded)

    autoencoder = Model(input_layer, output_layer)
    return autoencoder

In [11]:
autoencoder = create_autoencoder(input_dim=n_features)
autoencoder.compile(optimizer="adadelta", loss="mse")
autoencoder.summary()

# Fit the model
history = autoencoder.fit(
    X_train,
    X_train,
    batch_size=64,
    epochs=20,
    verbose=0,
    validation_split=0.15,
    callbacks=[TqdmCallback(), EarlyStopping(patience=5)],
)

# Predict reconstruction errors for the training set
reconstructions = autoencoder.predict(X_train)
train_loss = tf.keras.losses.mae(reconstructions, X_train).numpy()
threshold = np.mean(train_loss) + np.std(train_loss)
print("Threshold: ", threshold)

# Predict reconstruction errors for the test set
reconstructions = autoencoder.predict(X_test)
test_loss = tf.keras.losses.mae(reconstructions, X_test).numpy()
preds = predict(autoencoder, X_test, threshold)

# Calculate recall and confusion matrix
conf_matrix = confusion_matrix(y_test, preds)
tn, fp, fn, tp = conf_matrix.ravel()

precision_score = tp / (tp + fp)
recall = tp / (tp + fn)
fpr = fp / (fp + tn)

print(f"Precision: {precision_score}")
print(f"False Positive Rate: {fpr}")
print(conf_matrix)

# Save the recall and false positive rate
metrics.append(
    {
        "n_layers": 5,
        "Precision": precision_score,
        "Recall": recall,
        "false_positive_rate": fpr,
    }
)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Input (InputLayer)              │ (None, 359)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Encoding (Dense)                │ (None, 128)            │        46,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Latent (Dense)                  │ (None, 16)             │         2,064 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Decoding (Dense)                │ (None, 128)            │         2,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Output (Dense)                  │ (None, 359)            │        46,311 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 96,631 (377.46 KB)

 Trainable params: 96,631 (377.46 KB)

 Non-trainable params: 0 (0.00 B)

0epoch [00:00, ?epoch/s]

0batch [00:00, ?batch/s]

4579/4579 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step
Threshold:  0.0956957583340156
1145/1145 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step
1145/1145 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step
Precision: 0.9996223345534833
False Positive Rate: 0.04887218045112782
[[  253    13]
 [ 1956 34409]]
